# Churn prediction project

Here x is the feature vector - everything we know about the customer - and y is the target variable. The target is binary, so y belongs to {0, 1}:

1 is the positive class: the customer churned
0 is the negative class: the customer stayed

The output of the model - the score - is the likelihood of churning. In this project, 1 means the customer left the company and 0 means the customer stayed.

We learn g from historical data. We take the customers of the previous month - we already know what happened to them. For each of them we know the demographics, what services they use, what kind of contract they have and how much they pay, and we know whether they stayed or left. The model finds the patterns in this data, and then we apply it to the current customers to score them.

Data preparation

In this unit we download the churn dataset, read it with pandas and prepare it for machine learning: make the column names and values uniform, fix the column that pandas read incorrectly, and translate the target variable to numbers.

In [99]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [100]:
df = pd.read_csv('Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


To see all of them at once we can transpose the dataframe - flip rows and columns - with df.head().T

In [101]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [102]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Making everything uniform

The column names are inconsistent: some start with an uppercase letter (CustomerID, Tenure), some contain spaces (Monthly Charges is MonthlyCharges here, but the values inside columns do contain spaces). The values are also inconsistent - we see No, Yes, No phone service and so on. We lowercase everything and replace spaces with underscores:

In [103]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_variables = list(df.dtypes[df.dtypes == 'str'].index)     # .index returns the column names

for c in categorical_variables:
    df[c] = df[c].str.lower().str.replace(' ', '_')

Fixing the totalcharges column

Let's check the types that pandas inferred: df.dtypes

Two things stand out:

1. seniorcitizen is int64 - it is stored as 0/1 instead of yes/no. That's fine, we will treat it as a categorical variable later.
2. totalcharges is object - a string column - although it clearly contains numbers.

Let's try to convert it to numbers:

In [104]:
pd.to_numeric(df.totalcharges, errors='coerce') # force the conversion with errors='coerce'. Unparseable strings are then replaced with NaN

0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: totalcharges, Length: 7043, dtype: float64

It fails without errors='coerce:

ValueError: Unable to parse string "_" at position 488

The reason: in the original data, missing values in this column were encoded with a space. Our normalization step replaced spaces with underscores, so the missing values became "_", and pandas cannot parse that as a number.

In [105]:
# Save it to tc

tc = pd.to_numeric(df.totalcharges, errors='coerce')

Let's see how many missing values this creates:

In [106]:
tc.isnull().sum() # check how many NaN values we have

np.int64(11)

There are 11 such rows. We can inspect them:

In [107]:
df[tc.isnull()][['customerid', 'totalcharges']]

,customerid,totalcharges
488,4472-lvygi,_
753,3115-czmzd,_
936,5709-lvoeq,_
1082,4367-nuyao,_
1340,1371-dwpaz,_
3331,7644-omvmy,_
3826,3213-vvolg,_
4380,2520-sgtta,_
5218,2923-arzlg,_
6670,4075-wkniu,_


These are customers who just joined - their tenure is small and they haven't been billed yet, so their total charges are not available. For them we simply fill the missing values with zero:

In [108]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

Zero is not always the best replacement, but in this case it is acceptable: it matches the fact that these customers haven't paid anything yet.

Encoding the target variable

The target, churn, still contains strings:

In [109]:
# Checking the target variable

df.churn.head()

0     no
1     no
2    yes
3     no
4    yes
Name: churn, dtype: str

For classification we need numbers. We compare the column with 'yes'

this produces a boolean series - and cast it to integers:

In [110]:
df.churn = (df.churn == 'yes').astype(int)

In [111]:
df.churn.head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

# Setting up the validation framework

Before training anything, we split the data into train, validation and test sets using Scikit-Learn, and we put the target variable aside so it cannot leak into the features.

Splitting the data

We use train_test_split from sklearn.model_selection:

In [114]:
from sklearn.model_selection import train_test_split

We want the classic 60/20/20 split: 60% of the data for training, 20% for validation and 20% for testing. The function only splits in two, so we do it in two steps.

In [115]:
# Set aside the test set - 20% of all the data:

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

The remaining 80% we call the full train set. Then we split it again into train and validation:

In [120]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

Note that here we use 0.25, not 0.2. 

The second split gets only 80% of the original data, and we want the validation set to be 20% of the original. Since 20% of 80% is 25%, test_size=0.25 gives us exactly what we need - and that's why validation and test end up the same size:

In [121]:
len(df_train), len(df_val), len(df_test)

(4225, 1409, 1409)

The random_state=1 argument fixes the seed of the random shuffling. The split is random, but with a fixed seed it is reproducible - running the notebook again gives exactly the same split.

Resetting the index

train_test_split shuffles the rows, so after splitting the row indexes are a random mixture of numbers. We reset them:

In [122]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

The drop=True argument discards the old index instead of adding it back as a column. We do the same for the full train set when we start working with it.

Isolating the target

The target for this project is the churn column. We extract it from each dataframe as a NumPy array with .values:

In [123]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

Then we delete the column from the dataframes:

This is a safety measure: we don't want to accidentally use the target as a feature when we train the model. Note that we keep churn inside df_full_train for now - we need it in the next unit, where we do exploratory data analysis.

In [124]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

# EDA

Now we explore the training data: we check for missing values, look at the distribution of the target variable, and go through the numerical and categorical columns. For EDA we use the full train set - train and validation together - because we will also use it to compute feature importance later.

First we reset its index, as we did for the other parts:

In [126]:
df_full_train = df_full_train.reset_index(drop=True)

Missing values

The check is simple - for every column we count the NaN values:

In [127]:
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

Everything is 0 - there are no missing values anywhere. We already took care of the only column that had them, totalcharges, during data preparation.

Looking at the target: the churn rate

Next we look at the target variable - how many customers churned and how many stayed:

In [130]:
df_full_train.churn.value_counts(normalize=True)    # normalize=True converts to percentage, : Divides each count by the total number of values

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

About 73% of the customers stayed and 27% churned. The percentage of ones - 0.269968, roughly 27% - is called the churn rate.

Because churn is now a column of ones and zeros, there is a shortcut for computing it: the mean. 

The mean of a binary column is the number of ones divided by the total number of rows, which is exactly the churn rate:

In [131]:
df_full_train.churn.mean()

np.float64(0.26996805111821087)

Same number. This works for any binary variable: its mean is always the fraction of ones.

Numerical and categorical variables

Next we decide which columns are numerical and which are categorical. Most columns in this dataset contain strings, so they are categorical. Among the numeric ones, seniorcitizen is really a categorical variable - it is just encoded as 0/1 - and churn is the target. That leaves three numerical variables:

In [132]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

And 16 categorical ones - everything except customerid (an identifier, useless as a feature), the three numerical variables and the target:

In [133]:
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

It is a good idea to check how many unique values each categorical variable has:

In [135]:
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

Most variables have two or three values; paymentmethod has four - it is the most diverse one. The number of categories matters later: it determines how many columns we get after one-hot encoding.

# Feature importance: Churn rate and risk ratio

Feature importance analysis is a part of EDA: we want to identify which features affect our target variable. In this unit we do it for categorical variables by comparing the churn rate of each group with the global churn rate, first as a difference and then as a ratio.

Churn rate per group

The reference point is the global churn rate - the churn rate we computed in the EDA unit:

In [136]:
global_churn = df_full_train.churn.mean()